# **02477 Bayesian Machine Learning | Conceptual reasoning guide**



---

### **What this notebook is for**

PART 1–3 are formula references - we search them by keyword and find the equation. This notebook answers a different question: **given what the exam task says, which formula do we need, why, and how does it change when the scenario changes?**

Use this notebook when we find a formula in PART 1–3 but are unsure whether it applies, or when a small twist in the problem description makes we doubt what to reach for.

---

### **Table of Contents**


1. [Section 1 | Problem recognition - what model does this task imply?](#sec1)

2. [Section 2 | When is the answer analytical vs. when must we sample?](#sec2)

3. [Section 3 | The marginalisation decision tree](#sec3)

4. [Section 4 | Linear Gaussian systems - general joint construction](#sec4)

5. [Section 5 | Kernel questions - prior variance, scaling, stationarity](#sec5)

6. [Section 6 | Variational inference - complete worked example](#sec6)

7. [Section 7 | Credibility intervals and probabilities from any distribution](#sec7)

8. [Section 8 | Exam question taxonomy - what each question type is really asking](#sec8)


---

<a id='sec1'></a>

<div class="alert alert-block alert-warning">

### **Section 1 | Problem recognition - what model does this task imply?**

Read the task description. What is $y$? What is its domain? That determines the likelihood, link function, and everything downstream.
</div>

##### **1.1 The likelihood selection table**


The single most important question when reading an exam task is: **what type of variable is $y_n$?** Everything else follows from this.

<span style="color: blue;">

| Observed $y_n$ | Domain | Likelihood | Link / mean function | Keyword signals in task |
|---|---|---|---|---|
| Continuous real | $\mathbb{R}$ | $\mathcal{N}(\mu, \sigma^2)$ | Identity: $\mu = w^T\phi(x)$ | "regression", "continuous", "additive noise $\epsilon$" |
| Binary | $\{0,1\}$ | $\text{Bernoulli}(\sigma(f))$ | Sigmoid: $\mu = \sigma(w^T\phi(x))$ | "binary", "classification", "success/failure", $y \in \{0,1\}$ |
| Count | $\{0,1,2,\ldots\}$ | $\text{Poisson}(e^f)$ | Log: $\mu = \exp(w^T\phi(x))$ | "count", "events", "non-negative integer", $y \in \mathbb{Z}^+$ |
| Multi-class | $\{1,\ldots,K\}$ | $\text{Categorical}(\text{softmax}(f))$ | Softmax: $\pi_k = \frac{e^{f_k}}{\sum_j e^{f_j}}$ | "$K$ classes", "categories", $y \in \{1,\ldots,K\}$ |
| Positive real | $\mathbb{R}^+$ | $\text{Gamma}$ | Log: $\mu = \exp(f)$ | "duration", "waiting time", "positive" |

</span>



<span style="color: red;">

**The swap rule - what changes and what stays the same:**

> The linear predictor $f(x) = w^T\phi(x)$ **stays the same** regardless of likelihood.

> The Gaussian prior $p(w) = \mathcal{N}(w|0, \alpha^{-1}I)$ **stays the same**.

> The procedure (prior → MAP → Laplace → predictive) **stays the same**.

> Only the likelihood $p(y_n | f(x_n))$ **changes**. Everything else is plumbed in automatically.

</span>

##### **1.2 What "adapt the model" means on the exam**


When the exam says *"explain how we would adapt the model for predicting $z^*$ from $x^*$"* (e.g. 2025R Q2.5), the expected answer always has this structure:

**Step 1 - Identify what $z_n$ is.** Look at the domain: if $z_n \in \{0,1,2,\ldots\}$ it is a count → Poisson likelihood.

**Step 2 - State the new likelihood.** Write $p(z_n | w, x_n) = \text{Poisson}(\exp(w^T\phi(x_n)))$ (or whichever applies).

**Step 3 - Keep everything else.** Same prior $p(w)$, same MAP procedure, same Laplace approximation. Only the likelihood expression changes.

**Step 4 - State how the predictive changes.** The posterior predictive $p(z^* | y, x^*)$ is now $\int \text{Poisson}(z^* | \exp(w^T\phi(x^*))) \, q(w) \, dw$, which is intractable - so estimate it by sampling $w^{(s)} \sim q(w)$ and averaging $\text{Poisson}(z^* | \exp(w^{(s)T}\phi(x^*)))$.

> **Exam trap:** The question never says "write down the Poisson distribution" - it says "explain how to adapt". A full-mark answer names the new likelihood, the new link function, keeps the prior the same, and acknowledges that the posterior is now intractable (so requires Laplace or MCMC).

##### **1.3 Recognising heteroscedastic models**


**Signal in the task:** "the variance depends on the input", "input-dependent noise", "extend the model to capture heteroscedastic noise", or the likelihood is written $\mathcal{N}(y_n | f_1(x_n), e^{f_2(x_n)})$.

**What this means:** the network or linear model produces *two* outputs - one for the mean and one for the log-variance.

<span style="color: blue;">

$$p(y_n | w, x_n) = \mathcal{N}\!\left(y_n \,\Big|\, \underbrace{f_1(x_n|w)}_{\text{predicted mean}},\; \underbrace{e^{f_2(x_n|w)}}_{\text{predicted variance}}\right)$$

</span>

The log-likelihood for optimisation becomes:

<span style="color: blue;">

$$\log p(y_n|w,x_n) = -\tfrac{1}{2}f_2(x_n|w) - \frac{(y_n - f_1(x_n|w))^2}{2\,e^{f_2(x_n|w)}} + \text{const}$$

</span>

<span style="color: red;">

The $e^{f_2}$ parameterisation is necessary to guarantee $\sigma^2 > 0$ for any real-valued output $f_2$. The prior stays Gaussian on all weights $w = [w_1, w_2, w_3, w_4]^T$. MAP and Laplace proceed as usual - just more parameters.

</span>

**For variational inference on the heteroscedastic model** (e.g. 2025R Q4.5–4.6): the mean-field approximation treats each $w_i$ independently, so entropy and KL decompose as sums over $i$. When computing the predictive, sample $w^{(s)} \sim q(w)$, compute $f_1^{(s)} = f_1(x^*|w^{(s)})$ and $f_2^{(s)} = f_2(x^*|w^{(s)})$, then evaluate $\mathcal{N}(y^* | f_1^{(s)}, e^{f_2^{(s)}})$, then average over $s$.

##### **1.4 Recognising mixture models**


**Signal in the task:** a discrete latent variable $z_n \in \{0,1\}$ or $s \in \{0,1\}$ appears in the likelihood. The model looks like:

$$p(y_n | \theta, z_n) = (1-z_n)\,p_0(y_n) + z_n\,p_1(y_n|\theta)$$

or equivalently, the likelihood is a *weighted sum* of two Gaussians where the weight is determined by a Bernoulli variable.

**This is not the same as a standard likelihood model.** See Section 3 for how to marginalise out $z_n$.

> **Exam trap:** When we see $p(y|v, s)$ where $s \in \{0,1\}$, the exam is almost certainly going to ask we to compute $p(y|v)$ (marginalise out $s$) and then $p(y)$ (marginalise out both $s$ and $v$). Section 3 shows exactly how.

---

<a id='sec2'></a>

<div class="alert alert-block alert-warning">

### **Section 2 | When is the answer analytical vs. when must we sample?**

This is the question the notebooks never answer directly. Use this section to decide whether to reach for a formula or for sampling code.
</div>

##### **2.1 The tractability table**


<span style="color: blue;">

| Likelihood | Prior on $w$ | Posterior $p(w\|y)$ | Predictive $p(y^*\|y,x^*)$ | Method |
|---|---|---|---|---|
| Gaussian $\mathcal{N}(y\|w^T\phi, \sigma^2)$ | Gaussian $\mathcal{N}(w\mid 0,\alpha^{-1}I)$ | **Gaussian - exact** | **Gaussian - exact** | PART 1 §3.4 formulas directly |
| GP prior $f \sim \mathcal{GP}(0,k)$, Gaussian obs | - | **Gaussian - exact** | **Gaussian - exact** | PART 1 §5 formulas directly |
| Bernoulli $\text{Ber}(\sigma(f))$ | Gaussian | **Intractable** | **Intractable** | Laplace approx → probit |
| Poisson $\text{Pois}(e^f)$ | Gaussian | **Intractable** | **Intractable** | Laplace approx or MCMC |
| Categorical $\text{Cat}(\text{softmax}(f))$ | Gaussian | **Intractable** | **Intractable** | Laplace or MCMC or VI |
| Any non-linear model in $w$ | Any | **Intractable** | **Intractable** | MCMC (Metropolis) |
| Mixture prior or mixture likelihood | Gaussian | **Intractable in general** | **Intractable** | Marginalise analytically then check |

</span>




<span style="color: red;">

**Rule:** If both the likelihood AND the prior are Gaussian AND the mean of the likelihood is LINEAR in $w$, then the posterior is Gaussian and we have a closed-form formula. If any of those three conditions fails, we need an approximation.

</span>

##### **2.2.1 What "tractable" and "intractable" actually mean in practice**



**Tractable (closed-form)** means the posterior is a Gaussian whose mean $m$ and covariance $S$ we can write down directly using the formulas in PART 1. we never need to run any algorithm - just plug numbers into the formula and compute. The predictive is also Gaussian, so probabilities and intervals come from `norm.cdf` / `norm.ppf`.

**Intractable** means no formula exists for the posterior. we cannot write it down in closed form. we must *approximate* it using one of three strategies:

| Strategy | When to use it | What we get |
|---|---|---|
| **Laplace approximation** | Non-Gaussian likelihood, posterior has a single mode (unimodal) | A Gaussian $q(w) = \mathcal{N}(\hat{w}, H^{-1})$ centred at the MAP estimate |
| **MCMC (Metropolis)** | Any intractable posterior, especially non-linear models | A set of samples $w^{(1)}, \ldots, w^{(S)}$ approximately from the true posterior |
| **Variational inference (VI)** | Intractable posterior, but we want a fast deterministic approximation | A Gaussian $q^*(w)$ found by maximising the ELBO |

**After approximation - what we actually do:**

> **After Laplace:** treat $q(w) = \mathcal{N}(m, S)$ exactly like a tractable Gaussian posterior. Use `norm.cdf` for probabilities, `norm.ppf` for intervals, and the probit formula for GP/logistic classification predictive.

> **After MCMC:** we have samples. Every downstream quantity is a sample mean: `jnp.mean(f(samples))`. Probabilities are `jnp.mean(samples > c)`. Intervals are `jnp.percentile(samples, [5, 95])`. we never use `norm.cdf` on MCMC output unless we separately fit a Gaussian to the samples.

> **After VI:** treat $q^*(w) = \prod_i \mathcal{N}(m_i, v_i)$ like a tractable posterior. Use `norm.cdf` for analytical quantities, or draw samples from $q^*$ and use `jnp.mean(...)` for non-analytical ones (e.g. Poisson predictive).

**The sampling-vs-formula decision within a tractable posterior:**

Even when the posterior IS Gaussian (tractable), we must still decide whether the *predictive* has a closed form:

> **Gaussian likelihood + Gaussian posterior → predictive is Gaussian → use formula.**

> **Non-Gaussian likelihood (Bernoulli, Poisson) + any posterior → predictive has NO closed form → sample $w^{(s)} \sim q(w)$, evaluate likelihood per sample, average.**

> **Key example:** after running MCMC on a Poisson model, the posterior predictive $p(y^*|y,x^*)$ requires sampling $\mu^{*(s)} = \exp(w^{(s)T}\phi(x^*))$ and then sampling $y^{*(s)} \sim \text{Poisson}(\mu^{*(s)})$. we need two layers of sampling - one for $w$, one for $y^*$.

**One-line rule to decide what to do next:**

> Ask: *"Is the thing I want to compute a linear function of the posterior?"* If yes → formula. If no → samples.
>
> $\mathbb{E}[w]$: linear → `m` directly. $P(w_1 > 0)$: use `norm.cdf`. $\mathbb{E}[\sin(w_1 w_2)]$: non-linear → samples. $p(y^*=3 | y, x^*)$ under Poisson → samples.

##### **2.2 The non-linear model - always MCMC**


When the model has the form $y = f(x|w) + \epsilon$ where $f$ is **non-linear in $w$** (e.g. $f(x|w) = w_2 \tanh(w_1 x)$, or $f(x|w) = e^w$), the posterior $p(w|y)$ is **never** Gaussian regardless of the Gaussian prior.

**Signal in the task:** the function $f$ involves $\tanh$, $\exp$, $\sin$, products of parameters ($w_1 w_2$), or any other non-linearity in $w$.

**What to do:** Run Metropolis MCMC (see PART 2 §8). The `log_target` is always:

```python
def log_target(w):
    f = ...          # evaluate f(x|w) - the non-linear function
    log_lik   = log_npdf(y, f, sigma2)              # p(y|w)
    log_prior = log_npdf(w[0], 0, 1) + ...          # p(w) - sum over all parameters
    return log_lik + log_prior
```

**For the posterior predictive after MCMC:** for each sample $w^{(s)}$, compute $f^{(s)} = f(x^*|w^{(s)})$, then compute what is asked (mean, probability, interval) using `jnp.mean(...)` over the samples.

> **Exam trap:** $f(x^*|w^{(s)})$ is the *latent function value*, not the *observation*. If the question asks for $p(y^* | y, x^*)$, we must also sample from the observation noise: $y^{*{(s)}} = f^{(s)} + \epsilon^{(s)}$ with $\epsilon^{(s)} \sim \mathcal{N}(0, \sigma^2)$. If it asks for $p(f(x^*) > c)$, just use $f^{(s)}$ directly.

##### **2.3 The bilinear model - VI but not Laplace**


A special case that appears repeatedly: $y | w_1, w_2 \sim \mathcal{N}(w_1 w_2, \sigma^2)$. The product $w_1 w_2$ is non-linear in the joint parameter vector $(w_1, w_2)$, so the posterior is **not Gaussian**.

For this model, the exam typically uses **mean-field variational inference** with $q(w_1, w_2) = \mathcal{N}(w_1|m_1, v_1)\mathcal{N}(w_2|m_2, v_2)$.

The key computation is $\mathbb{E}_q[w_1 w_2]$ and $\mathbb{E}_q[(w_1 w_2)^2]$:

<span style="color: blue;">

$$\mathbb{E}_q[w_1 w_2] = m_1 m_2 \qquad \text{(independence under mean-field)}$$

$$\mathbb{E}_q[w_1^2] = m_1^2 + v_1, \qquad \mathbb{E}_q[w_2^2] = m_2^2 + v_2$$

$$\mathbb{E}_q[(w_1 w_2)^2] = (m_1^2 + v_1)(m_2^2 + v_2)$$

$$\mathbb{E}_q[(y - w_1 w_2)^2] = y^2 - 2y\,m_1 m_2 + (m_1^2+v_1)(m_2^2+v_2)$$

</span>

<span style="color: red;">

The step $\mathbb{E}_q[w_1 w_2] = m_1 m_2$ uses independence of $w_1$ and $w_2$ under the mean-field approximation. This would NOT hold for the true posterior (they are correlated). The approximate posterior covariance between $w_1$ and $w_2$ under the optimal mean-field approximation is therefore **zero by construction** - this is the key limitation of mean-field VI.

</span>

<a id='sec3'></a>

<div class="alert alert-block alert-warning">

### **Section 3 | The marginalisation decision tree**

When to sum, when to integrate, and how to recognise which integral is tractable.
</div>

##### **3.1 The fundamental rule**


Marginalisation removes a variable we don't want by integrating (continuous) or summing (discrete) over all its values:

<span style="color: blue;">

$$p(y) = \int p(y, \theta)\, d\theta \qquad \text{(continuous } \theta \text{)}$$

$$p(y) = \sum_{z \in \{0,1\}} p(y, z) \qquad \text{(discrete } z \text{)}$$

</span>

**When the exam asks:** $p(y|v)$, $p(y, \theta)$, $p(y, z)$, $p(y)$ - we are being asked to marginalise something out.

##### **3.2 The decision tree - step by step**


**Step 1 - Identify what to marginalise.** What variable appears in the joint but not in the target?

> If it is **discrete** (e.g. $z \in \{0,1\}$, $s \in \{0,1\}$): **sum** over its values.

> If it is **continuous Gaussian** (e.g. $\theta \sim \mathcal{N}$, $v \sim \mathcal{N}$): use the **Gaussian marginalisation identity** (see Step 3).

> If it is **continuous non-Gaussian**: the integral is generally intractable - use VI or MCMC.

**Step 2 - Discrete variable: sum over the two terms.** For $z \in \{0,1\}$:

<span style="color: blue;">

$$p(y, \theta) = p(y, \theta, z{=}0) + p(y, \theta, z{=}1)$$
$$= p(y|\theta, z{=}0)\,p(z{=}0)\,p(\theta) + p(y|\theta, z{=}1)\,p(z{=}1)\,p(\theta)$$
$$= p(\theta)\Big[(1-\pi)\,p_0(y) + \pi\,p_1(y|\theta)\Big]$$

</span>

<span style="color: red;">

Here $\pi = p(z=1)$ is the Bernoulli mixing weight. After summing, $\theta$ may still be present - if so, we may need Step 3 to also integrate out $\theta$.

</span>

**Step 3 - Continuous Gaussian variable: use the marginalisation identity.** For $\theta \sim \mathcal{N}(0, \alpha^{-1}I)$ and $y | \theta \sim \mathcal{N}(A\theta, \Sigma)$:

<span style="color: blue;">

$$\int \mathcal{N}(y | A\theta, \Sigma)\,\mathcal{N}(\theta | 0, \alpha^{-1}I)\, d\theta = \mathcal{N}(y | 0,\; \Sigma + A(\alpha^{-1}I)A^T)$$

</span>

For the common exam case $y | \theta \sim \mathcal{N}(\theta^T x, \sigma^2)$ (scalar $y$, $A = x^T$):

<span style="color: blue;">

$$\int \mathcal{N}(y | \theta^T x, \sigma^2)\,\mathcal{N}(\theta | 0, \alpha^{-1}I)\, d\theta = \mathcal{N}(y | 0,\; \alpha^{-1}\|x\|^2 + \sigma^2)$$

</span>

##### **3.3 Full worked example (2025R Part 3 pattern)**


Model: $p(y|v,s) = \mathcal{N}(y|v, \sigma^2 + s\tau^2)$, $\;p(v) = \mathcal{N}(v|1,1)$, $\;p(s) = \text{Ber}(s|p)$, with $\sigma^2=1$, $\tau^2=2$, $p=\frac{1}{4}$.

**Task: find $p(y|v)$** - marginalise out discrete $s$:

$$p(y|v) = \sum_{s \in \{0,1\}} p(y|v,s)\,p(s)$$
$$= p(y|v,s{=}0)\,p(s{=}0) + p(y|v,s{=}1)\,p(s{=}1)$$
$$= \tfrac{3}{4}\,\mathcal{N}(y|v,1) + \tfrac{1}{4}\,\mathcal{N}(y|v,3)$$

This is a **Gaussian mixture** - it cannot be simplified to a single Gaussian.

**Task: find $p(y)$** - now also marginalise out continuous $v \sim \mathcal{N}(1,1)$:

$$p(y) = \int p(y|v)\,p(v)\,dv = \tfrac{3}{4}\int \mathcal{N}(y|v,1)\,\mathcal{N}(v|1,1)\,dv + \tfrac{1}{4}\int \mathcal{N}(y|v,3)\,\mathcal{N}(v|1,1)\,dv$$

Each integral uses the marginalisation identity with $A=1$, $\Sigma = \sigma_s^2$, $\mu_v=1$, $\Lambda_v=1$:

$$\int \mathcal{N}(y|v,\sigma_s^2)\,\mathcal{N}(v|1,1)\,dv = \mathcal{N}(y|1,\; \sigma_s^2 + 1)$$

So:

$$p(y) = \tfrac{3}{4}\,\mathcal{N}(y|1,2) + \tfrac{1}{4}\,\mathcal{N}(y|1,4)$$

**Task: find $p(s{=}1|y)$** - posterior of the discrete variable:

$$p(s{=}1|y) = \frac{p(y|s{=}1)\,p(s{=}1)}{p(y)} = \frac{p(y,s{=}1)}{p(y,s{=}0)+p(y,s{=}1)}$$

Where $p(y,s{=}1) = p(s{=}1)\int p(y|v,s{=}1)p(v)dv = \tfrac{1}{4}\,\mathcal{N}(y|1,4)$, and $p(y,s{=}0) = \tfrac{3}{4}\,\mathcal{N}(y|1,2)$.

> **Key insight:** The result $p(y)$ is a mixture of Gaussians, **not a single Gaussian**. we cannot collapse it further. The exam may ask we to evaluate it numerically at a specific $y$.

##### **3.4 Mixture prior - the non-conjugate case (2024R Part 3 pattern)**


When the **prior** is a mixture of Gaussians (e.g. $p(\theta) = \frac{1}{2}\mathcal{N}(\theta|-m,\tau^2 I) + \frac{1}{2}\mathcal{N}(\theta|m,\tau^2 I)$) and the likelihood is Gaussian:

**The marginal likelihood $p(y)$** is computed by applying the marginalisation identity to each component separately:

$$p(y) = \frac{1}{2}\int \mathcal{N}(y|X\theta,\sigma^2 I)\,\mathcal{N}(\theta|-m,\tau^2 I)\,d\theta + \frac{1}{2}\int \mathcal{N}(y|X\theta,\sigma^2 I)\,\mathcal{N}(\theta|m,\tau^2 I)\,d\theta$$

Each integral is a Gaussian with different mean but same covariance structure:
$$= \frac{1}{2}\mathcal{N}(y|-Xm,\; \sigma^2 I + \tau^2 XX^T) + \frac{1}{2}\mathcal{N}(y|Xm,\; \sigma^2 I + \tau^2 XX^T)$$

**The posterior $p(\theta|y)$** uses Bayes' rule with this mixture marginal as the denominator. It remains a mixture of Gaussians - it does **not** reduce to a single Gaussian.

> **Exam trap:** When the task says "compute the posterior density at $\theta = 0$", plug into $p(\theta|y) = p(y|\theta)p(\theta)/p(y)$ numerically - do not try to find a closed-form posterior distribution.

<a id='sec4'></a>

<div class="alert alert-block alert-warning">

### **Section 4 | Linear Gaussian systems - general joint construction**

How to build the joint from scratch for any chain, including non-standard ones.
</div>

##### **4.1 The three propagation rules**


For any linear Gaussian chain $v = Cu + \epsilon$ with $u \perp \epsilon$:

<span style="color: blue;">

| Quantity | Formula | Notes |
|---|---|---|
| Mean of $v$ | $\mathbb{E}[v] = C\,\mathbb{E}[u] + \mathbb{E}[\epsilon]$ | $\mathbb{E}[\epsilon]=0$ usually |
| Variance of $v$ | $\text{Cov}(v) = C\,\text{Cov}(u)\,C^T + \text{Cov}(\epsilon)$ | Add noise covariance |
| Cross-covariance | $\text{Cov}(u, v) = \text{Cov}(u)\,C^T$ | Always this form |

</span>



<span style="color: red;">

The cross-covariance rule $\text{Cov}(u,v) = \text{Cov}(u)\,C^T$ holds because $u$ and $\epsilon$ are independent, so $\text{Cov}(u, Cu + \epsilon) = \text{Cov}(u, Cu) = \text{Cov}(u)\,C^T$.

</span>

##### **4.2 The general construction procedure**


**Given any chain of equations, do this:**

**Step 1 - Propagate means.** Apply $\mathbb{E}[\cdot]$ through each equation in order.

**Step 2 - Propagate variances.** Apply the variance rule $\text{Cov}(v) = C\,\text{Cov}(u)\,C^T + \text{Cov}(\epsilon)$ in order.

**Step 3 - Compute all cross-covariances.** For every pair of variables in the chain, use $\text{Cov}(u,v) = \text{Cov}(u)\,C^T$.

**Step 4 - Stack into the joint.** Write the full joint Gaussian with the block covariance matrix.

**Step 5 - Apply conditioning formula** (PART 3 Bonus B) to get any conditional we need.

##### **4.3 Worked example (2025R Part 1 pattern)**


Model: $z = Ax + b + n_1$, $\;y = Wz + n_2$, with:
$$x \sim \mathcal{N}(0,I), \quad n_1 \sim \mathcal{N}(0,\tfrac{1}{2}I), \quad n_2 \sim \mathcal{N}(0,\tfrac{1}{2}I), \quad A=\begin{bmatrix}0&1\\1&0\end{bmatrix}, \; b=\begin{bmatrix}1\\1\end{bmatrix}, \; W=\begin{bmatrix}1&1\\1&0\end{bmatrix}$$

**Step 1 - Means:**
$$\mathbb{E}[z] = A\cdot 0 + b = b = \begin{bmatrix}1\\1\end{bmatrix}$$
$$\mathbb{E}[y] = W\,\mathbb{E}[z] = Wb$$

**Step 2 - Variances:**
$$\text{Cov}(z) = A\cdot I\cdot A^T + \tfrac{1}{2}I = AA^T + \tfrac{1}{2}I$$
$$\text{Cov}(y) = W\,\text{Cov}(z)\,W^T + \tfrac{1}{2}I$$

**Step 3 - Cross-covariances:**
$$\text{Cov}(x,z) = I\cdot A^T = A^T \qquad \text{(from } z = Ax + b + n_1 \text{, } C=A \text{)}$$
$$\text{Cov}(z,y) = \text{Cov}(z)\,W^T \qquad \text{(from } y = Wz + n_2 \text{, } C=W \text{)}$$

**Step 4 - Answer each sub-question by reading off from the joint:**

> $p(z|x)$: Gaussian with mean $Ax+b$ and covariance $\tfrac{1}{2}I$ (just read off the conditional definition).

> $p(x|z)$: Apply the conditioning formula to the joint $(x,z)$.

> $p(y)$: Gaussian with mean $Wb$ and covariance $W(AA^T+\tfrac{1}{2}I)W^T + \tfrac{1}{2}I$.

> **Key check:** $p(z|x)$ is immediate from the model equation itself - mean $Ax+b$, noise $\tfrac{1}{2}I$. we do NOT need to go through the full joint construction for this sub-question.

##### **4.4 The Markov chain variant (2024 Part 1 pattern)**


Model: $x_1 \sim \mathcal{N}(0,\Sigma)$, $\;x_2|x_1 \sim \mathcal{N}(Ax_1,\Sigma)$, $\;x_3|x_2 \sim \mathcal{N}(Ax_2,\Sigma)$.

<span style="color: blue;">

| What is asked | How to compute | Result |
|---|---|---|
| $p(x_2)$ | Apply variance propagation to $x_2 = Ax_1 + \epsilon$ | $\mathcal{N}(0,\; A\Sigma A^T + \Sigma)$ |
| $p(x_3)$ | Apply again: $x_3 = Ax_2 + \epsilon$, using $\text{Cov}(x_2)$ | $\mathcal{N}(0,\; A^2\Sigma A^{2T} + A\Sigma A^T + \Sigma)$ |
| $p(x_3\|x_1)$ | $x_3 = A^2 x_1 + A\epsilon_1 + \epsilon_2$ | $\mathcal{N}(A^2 x_1,\; A\Sigma A^T + \Sigma)$ |
| $p(x_1\|x_2)$ | Build joint $(x_1, x_2)$, apply conditioning formula | Use $\text{Cov}(x_1,x_2) = \Sigma A^T$ |

</span>

**For $p(x_1|x_2)$** - the joint is:
$$\begin{bmatrix}x_1\\x_2\end{bmatrix} \sim \mathcal{N}\!\left(0,\; \begin{bmatrix}\Sigma & \Sigma A^T \\ A\Sigma & A\Sigma A^T+\Sigma\end{bmatrix}\right)$$

Then apply the conditioning formula from PART 3 Bonus B directly.

<a id='sec5'></a>

<div class="alert alert-block alert-warning">

### **Section 5 | Kernel questions - prior variance, scaling, stationarity**

What each kernel-related exam question is actually testing.
</div>

##### **5.1 Prior variance at a test point**


**Signal in the task:** "determine the prior distribution $p(f^*)$", "what is the prior variance of $f(x^*)$?", "determine $p(f^*|x^*)$ analytically".

**Answer:** Under a GP prior $f \sim \mathcal{GP}(0, k)$, any single function value $f(x^*)$ is marginally Gaussian:

<span style="color: blue;">

$$p(f^*) = \mathcal{N}(f^* \mid 0,\; k(x^*, x^*))$$

</span>

No data. No posterior formula. Just evaluate the kernel at the test point with itself.

**For observation $y^* = f(x^*) + \epsilon$:** add noise variance:
$$p(y^*|x^*) = \mathcal{N}(y^* \mid 0,\; k(x^*,x^*) + \sigma^2)$$

> **Exam trap:** The question says "prior" - that means no data, no posterior formula. Many students reach for the GP posterior formula and get confused about what to plug in. If there is no data yet, just evaluate $k(x^*,x^*)$.

##### **5.2 Reading off hyperparameters from a kernel expression**


The squared exponential (SE) kernel in standard form is:

<span style="color: blue;">

$$k(x,x') = \kappa^2 \exp\!\left(-\frac{\|x-x'\|^2}{2\ell^2}\right)$$

</span>

When the exam writes it in a different form, pattern-match the coefficients:

| Exam writes | Match to standard form | Read off |
|---|---|---|
| $2\exp(-\frac{1}{8}\|x-x'\|^2)$ | $\kappa^2 = 2$, $\frac{1}{2\ell^2} = \frac{1}{8}$ | $\kappa=\sqrt{2}$, $\ell=2$ |
| $5(1 + \exp(-\frac{1}{4}(x-x')^2))$ | Not SE - this is a *different* kernel | - |
| $\exp(-\frac{1}{2}\|x-x'\|^2) + 2$ | $k_1$ (SE with $\kappa=1$, $\ell=1$) $+ k_2$ (constant 2) | Sum kernel |

**Prior variance from the kernel:** $k(x^*,x^*) = \kappa^2 \cdot \exp(0) = \kappa^2$. So the prior variance under the SE kernel is just $\kappa^2$ regardless of $x^*$. For a constant kernel $+c$: prior variance = $\kappa^2 + c$.

##### **5.3 What scaling a kernel by a constant does**


**Signal in the task:** "suppose the kernel is changed from $k_1$ to $k_2 = c\,k_1$, explain how the posterior predictive changes" (e.g. 2025 Q4.5 where $k_2 = \frac{1}{10}k_1$).

**Effect of scaling by $c < 1$:**

> Prior variance $k_2(x^*,x^*) = c\,k_1(x^*,x^*)$ - **smaller prior variance**, GP functions have smaller amplitude.

> Prior kernel matrix $K_2 = c\,K_1$ - scaled down.

> The posterior mean $\mu_{f^*} = k_{2,*}^T(K_2 + \sigma^2 I)^{-1}y = c\,k_{1,*}^T(cK_1 + \sigma^2 I)^{-1}y$ - **pulled toward zero** (the prior dominates more).

> The posterior variance $\sigma^2_{f^*}$ - **reduced** because the prior is tighter.

> **Net effect on GP classification:** $\mu_{f^*}$ shrinks toward zero and $\sigma^2_{f^*}$ shrinks - the probit formula $\Phi(\mu_{f^*}/\sqrt{8/\pi + \sigma^2_{f^*}})$ moves toward $\Phi(0) = 0.5$. **The prediction becomes less confident and moves toward 0.5.**

> **The length-scale does NOT change.** Only the amplitude (prior variance) changes. The relative smoothness of functions is unchanged.

##### **5.4 Stationarity and isotropy - the two-step check**


<span style="color: blue;">

| Property | Definition | How to check | Non-stationary signals |
|---|---|---|---|
| **Stationary** | $k(x,x')$ depends only on $x-x'$ | Can we rewrite it as $k(x-x')$? | Terms like $xx'$, $x^2$, $x'^2$ appear separately |
| **Isotropic** | $k(x,x')$ depends only on $\|x-x'\|$ | Can we rewrite it as $k(\|x-x'\|)$? | Directional structure (e.g. ARD kernels) |

</span>

**Rule for sum/product kernels:**

> Sum: stationary iff **both** summands are stationary.

> Product: stationary iff **both** factors are stationary.

**Common exam example** - $k_2(x,x') = c_1\left(1 + \frac{\|x-x'\|}{2\ell}\right)^{-1} + c_2 x x'$:

> First term: depends on $\|x-x'\|$ → stationary and isotropic ✓

> Second term: $c_2 x x'$ - this is NOT expressible as a function of $x-x'$ alone → **not stationary** ✗

> Therefore $k_2$ is **not stationary** and hence also **not isotropic**.

<a id='sec6'></a>

<div class="alert alert-block alert-warning">

### **Section 6 | Variational inference - complete worked example**

The full pipeline from ELBO to predictive distribution, with every formula used in context.
</div>

##### **6.1 The ELBO decomposition - which term is which**


<span style="color: blue;">

$$\mathcal{L}[q] = \underbrace{\mathbb{E}_q[\log p(y|w)]}_{\text{expected log-likelihood}} - \underbrace{\text{KL}[q(w)\,\|\,p(w)]}_{\text{KL from prior}}$$

</span>

An equivalent decomposition (which the exam sometimes uses) is:

$$\mathcal{L}[q] = \mathbb{E}_q[\log p(y|w)] + H[q] - \mathbb{E}_q[-\log p(w)]$$

or written out:

<span style="color: blue;">

$$\mathcal{L}[q] = \mathbb{E}_q[\log p(y|w)] + \mathbb{E}_q[\log p(w)] + H[q]$$

</span>

<span style="color: red;">

| Term | What it penalises | How to compute it |
|---|---|---|
| $\mathbb{E}_q[\log p(y\|w)]$ | Data fit - how well $q$ predicts the data on average | Expand the log-likelihood, use $\mathbb{E}_q[w]=m$, $\mathbb{E}_q[w^2]=m^2+v$ |
| $\text{KL}[q\|p]$ | Deviation from prior - penalises $q$ for being far from $p(w)$ | Use the KL formula for Gaussians (see 6.2) |
| $H[q]$ | Entropy of $q$ - rewards $q$ for being spread out (uncertainty) | $\frac{1}{2}\ln(2\pi e\,v)$ per dimension |

</span>

##### **6.2 KL divergence for Gaussian mean-field families**


For a single Gaussian vs. standard Gaussian prior:

<span style="color: blue;">

$$\text{KL}[\mathcal{N}(m,v)\,\|\,\mathcal{N}(0,1)] = \frac{1}{2}\left(v + m^2 - 1 - \ln v\right)$$

</span>

For a mean-field family $q(w) = \prod_i \mathcal{N}(w_i|m_i,v_i)$ vs. prior $p(w) = \prod_i \mathcal{N}(w_i|0,1)$:

<span style="color: blue;">

$$\text{KL}[q\|p] = \sum_i \text{KL}[\mathcal{N}(m_i,v_i)\,\|\,\mathcal{N}(0,1)] = \frac{1}{2}\sum_i \left(v_i + m_i^2 - 1 - \ln v_i\right)$$

</span>

**When the prior is $\mathcal{N}(0, \alpha^{-1})$ (not standard normal):** rescale by dividing variance by $\alpha^{-1}$:

$$\text{KL}[\mathcal{N}(m,v)\,\|\,\mathcal{N}(0,\alpha^{-1})] = \frac{1}{2}\left(\alpha v + \alpha m^2 - 1 - \ln(\alpha v)\right)$$

> **When to use:** whenever the exam gives us variational parameters $m_i$, $v_i$ and asks we to evaluate the ELBO, the entropy, or the KL from the prior. The KL and entropy terms always decompose into sums over $i$ under mean-field.

##### **6.3 Entropy of a mean-field Gaussian**


<span style="color: blue;">

$$H[q] = H\!\left[\prod_i \mathcal{N}(m_i, v_i)\right] = \sum_i \frac{1}{2}\ln(2\pi e\, v_i)$$

</span>

**Numerical evaluation:** for each $i$, compute $\frac{1}{2}\ln(2\pi e\, v_i)$ and sum.

Example (2025R Q4.5): $v = [0.38, 0.19, 0.13, 0.09]^T$:
$$H[q] = \sum_{i=1}^4 \frac{1}{2}\ln(2\pi e\, v_i) = \frac{1}{2}\left[\ln(2\pi e \cdot 0.38) + \ln(2\pi e \cdot 0.19) + \ln(2\pi e \cdot 0.13) + \ln(2\pi e \cdot 0.09)\right]$$

```python
import jax.numpy as jnp
v = jnp.array([0.38, 0.19, 0.13, 0.09])
H = 0.5 * jnp.sum(jnp.log(2 * jnp.pi * jnp.e * v))
```

##### **6.4 Computing the posterior predictive under VI**


**Signal in the task:** "evaluate the posterior predictive density $p(y^*|y,x^*)$ using the variational approximation $q(w)$ and Monte Carlo estimation".

**Procedure:**

1. Sample $w^{(s)} \sim q(w)$ for $s = 1, \ldots, S$. Under mean-field: sample each $w_i^{(s)} \sim \mathcal{N}(m_i, v_i)$ independently.

2. For each sample, evaluate the likelihood at the test point: $p(y^*|w^{(s)}, x^*) = \mathcal{N}(y^*|f_1(x^*|w^{(s)}), e^{f_2(x^*|w^{(s)})})$.

3. Average: $p(y^*|y,x^*) \approx \frac{1}{S}\sum_s p(y^*|w^{(s)},x^*)$.

```python
from scipy.stats import norm
import jax.numpy as jnp
from jax import random

key = random.PRNGKey(0)
S = 10000

# Mean-field: sample each w_i independently
mu  = jnp.array([-0.82, 2.37, -0.88, 0.37])     # variational means
v   = jnp.array([0.38,  0.19,  0.13, 0.09])     # variational variances
eps = random.normal(key, shape=(S, 4))
w_samples = mu + jnp.sqrt(v) * eps              # shape (S, 4)

# Evaluate likelihood for each sample at test point x_star=1, y_star=1
x_star, y_star = 1.0, 1.0
f1 = w_samples[:,0] + w_samples[:,1] * x_star               # predicted mean
f2 = w_samples[:,2] + w_samples[:,3] * x_star               # log predicted variance
log_p = norm.logpdf(y_star, loc=f1, scale=jnp.exp(0.5*f2))  # log p(y*|w^(s),x*)

# Monte Carlo estimate of p(y*|y,x*)
p_predictive = jnp.exp(jnp.log(jnp.mean(jnp.exp(log_p))))
```

> **Note:** Use `log-sum-exp` trick for numerical stability when averaging in log space. The exam usually accepts the direct average.

##### **6.5 The KL decreases when we enlarge the variational family**


**Signal in the task:** "explain how $\text{KL}[q^*_1\|p]$ changes when switching from mean-field $Q_1$ to full-rank Gaussian $Q_2$" (e.g. 2025 Q3.6).

**Reasoning:**

> $Q_1$ (mean-field) is a subset of $Q_2$ (full-rank Gaussians) - because a diagonal covariance is a special case of a full covariance matrix.

> Therefore the minimum KL over $Q_2$ is **at most** as large as the minimum KL over $Q_1$. Any solution that was optimal under $Q_1$ is also feasible under $Q_2$, and $Q_2$ has more freedom to get closer to the true posterior.

> **$\text{KL}[q^*_2\|p] \leq \text{KL}[q^*_1\|p]$** - the KL can only decrease or stay the same.

> The inequality is strict if the true posterior has off-diagonal correlations (which a non-linear model typically does).

**General principle:** a larger variational family always gives a better (lower) KL divergence, because it contains all the solutions of the smaller family and potentially better ones.

<a id='sec7'></a>

<div class="alert alert-block alert-warning">

### **Section 7 | Credibility intervals and probabilities from any distribution**

The same conceptual operation appears in many different forms. This section unifies them.
</div>

##### **7.1 The universal probability/interval recipe**


All of these exam questions are asking the same thing - just phrased differently:

> "determine the prior probability of the event $w_1 > 0$"

> "compute a 90% credibility interval for $w_2$"

> "determine the posterior probability $p(w_1 > 0 | y)$"

<span style="color: blue;">

| If the distribution is... | Use... | Python |
|---|---|---|
| $\mathcal{N}(\mu, \sigma^2)$ **analytically** | `norm.cdf` / `norm.ppf` | `from scipy.stats import norm` |
| Multivariate $\mathcal{N}(m, S)$, marginal of $w_i$ | $w_i \sim \mathcal{N}(m_i, S_{ii})$ - read diagonal | `norm.cdf(0, m[i], jnp.sqrt(S[i,i]))` |
| From **MCMC samples** | `jnp.mean(samples > c)` or `jnp.percentile` | See below |
| From **prior** (no data) | Use prior parameters directly | Same formulas, just with prior $m$, $S$ |

</span>

##### **7.2 Probabilities from a Gaussian analytically**


For $w_i \sim \mathcal{N}(\mu_i, \sigma_i^2)$:

<span style="color: blue;">

$$P(w_i > 0) = 1 - \Phi\!\left(\frac{0 - \mu_i}{\sigma_i}\right) = \Phi\!\left(\frac{\mu_i}{\sigma_i}\right)$$

$$P(w_i > c) = 1 - \Phi\!\left(\frac{c - \mu_i}{\sigma_i}\right)$$

</span>

```python
from scipy.stats import norm
# P(w_i > 0) when w_i ~ N(mu_i, sigma_i^2)
prob = 1 - norm.cdf(0, loc=mu_i, scale=sigma_i)   # sigma_i is std, not variance!
# or equivalently:
prob = norm.cdf(mu_i / sigma_i)  # standardise first
```

**For a marginal of a multivariate Gaussian** $p(w|y) = \mathcal{N}(m,S)$:
$$p(w_i|y) = \mathcal{N}(w_i | m_i, S_{ii})$$
So $P(w_i > 0 | y) = \Phi(m_i / \sqrt{S_{ii}})$.

##### **7.3 Credibility intervals from a Gaussian analytically**


For $w_i \sim \mathcal{N}(\mu_i, \sigma_i^2)$, the symmetric $(1-\alpha)\times 100\%$ credibility interval is:

<span style="color: blue;">

$$[\mu_i - z_{\alpha/2}\,\sigma_i,\; \mu_i + z_{\alpha/2}\,\sigma_i]$$

</span>

| Confidence level | $\alpha$ | $z_{\alpha/2}$ | Python |
|---|---|---|---|
| 95% | 0.05 | 1.960 | `norm.ppf(0.975)` |
| 90% | 0.10 | 1.645 | `norm.ppf(0.95)` |
| 80% | 0.20 | 1.282 | `norm.ppf(0.90)` |

```python
from scipy.stats import norm
# 80% credibility interval for w_i ~ N(m_i, S_ii)
z = norm.ppf(0.90)   # = 1.2816
lower = m_i - z * jnp.sqrt(S_ii)
upper = m_i + z * jnp.sqrt(S_ii)
```

##### **7.4 Probabilities and intervals from samples**


When the distribution has no closed form (MCMC, non-linear model, Poisson predictive), use samples:

```python
# All samples[:,0] = w1, samples[:,1] = w2, etc. (in the order we defined them)

# Posterior probability P(w1 > 0 | y)
jnp.mean(samples[:,0] > 0)

# Posterior probability P(mu* > 7 | y) where mu* = exp(3 + w1*x* + w2*x*^2)
mu_star = jnp.exp(3 + samples[:,0]*x_star + samples[:,1]*x_star**2)
jnp.mean(mu_star > 7)

# 90% credibility interval (5th and 95th percentiles)
jnp.percentile(mu_star, jnp.array([5., 95.]))

# 95% credibility interval for y* ~ Poisson(mu*)
y_star_samples = jnp.array(np.random.poisson(np.array(mu_star)))
jnp.percentile(y_star_samples, jnp.array([2.5, 97.5]))
```

> **Exam trap on Poisson predictive:** $\mu^* = \exp(\ldots)$ gives the Poisson rate - this is NOT $y^*$ itself. To get $y^*$, sample from Poisson($\mu^*$) for each $\mu^{*(s)}$. The Poisson sample needs `np.random.poisson` (numpy, not jax), because JAX's Poisson sampler requires integer dtype handling.

<a id='sec8'></a>

<div class="alert alert-block alert-warning">

### **Section 8 | Exam question taxonomy - what each question type is really asking**

A map from question phrasing to the procedure that answers it, drawn from all four exam papers.
</div>

##### **8.1 Question type map**


<span style="color: blue;">

| Question phrasing | What it is asking | Where to look | Common trap |
|---|---|---|---|
| "Determine $p(y\|w,\Phi,\sigma^2)$" | Write down the likelihood (full dataset) | PART 1 §3.0 | Forgetting to write as $\mathcal{N}(y|\Phi w, \sigma^2 I)$ - the product over $n$ becomes a single Gaussian |
| "Compute the MLE for $w$" | Apply the normal equations | PART 1 §3.0.5 | Forgetting $\hat{w}_{\text{MLE}} = (\Phi^T\Phi)^{-1}\Phi^T y$; $\sigma^2$ is NOT needed |
| "Determine $p(w\|y)$" | Bayesian linear regression posterior - closed form | PART 1 §3.4 | Using MLE formula instead of posterior formula |
| "Determine $p(y^*\|y,x^*)$" | Posterior predictive - depends on the model | PART 1 §3.4 or GP §5 | For non-Gaussian: must use samples, not closed form |
| "Prior probability of $w_1 > 0$" | Evaluate $P(w_1 > 0)$ under $p(w) = \mathcal{N}(0, \alpha^{-1})$ | Section 7.2 this notebook | Prior has mean 0 → answer is exactly 0.5 if prior is symmetric |
| "Posterior probability of $w_1 > 0$" | Evaluate $P(w_1>0)$ under posterior marginal | Section 7.2 | Must use posterior mean $m_1$ and posterior variance $S_{11}$ |
| "80% credibility interval for $w_2$" | Compute interval under posterior marginal | Section 7.3 | 80% → use $z_{0.10} = 1.282$, not $1.96$ |
| "Verify $\hat{w}_{\text{MAP}}$ is correct" | Gradient = 0, Hessian negative definite at $\hat{w}$ | PART 3 Bonus A §A.4 | Need to check BOTH gradient and Hessian |
| "Plug-in approximation using MAP" | Substitute $\hat{w}_{\text{MAP}}$ as if it were the true $w$ | PART 1 §3.3 | No uncertainty in $w$ - do NOT add $\phi^T S \phi$ term |
| "Laplace approximation predictive" | Uses posterior mean $m$ AND covariance $S$ from Laplace | PART 1 §3.4, §6.2 | Must include $\phi^T S \phi$ term (parameter uncertainty) |
| "Prior predictive $p(y^*\|x^*)$" | No data - just prior × likelihood marginalised | Section 5.1 for GP; §C.3 PART 3 for BLR | Do NOT use posterior formulas |
| "Prior predictive $p(y^*{=}0\|x^*)$ for GP classifier" | Probit with $\mu_{f^*}=0$, $\sigma^2_{f^*}=k(x^*,x^*)$ | Section 5.1 this notebook | Answer is always 0.5 when prior mean is 0 |
| "Determine the entropy of $q(w)$" | Sum of $\frac{1}{2}\ln(2\pi e\,v_i)$ over all dimensions | Section 6.3 this notebook | Forgetting the $e$ in $2\pi e\,v_i$ |
| "Determine the marginal likelihood" | Integrate out $w$ from the joint | PART 1 §3.4 for BLR; Section 3 for mixtures | For mixture prior → result is a mixture, not single Gaussian |
| "Run Metropolis, plot traces" | Implement `log_target`, call sampler, discard warmup, plot | PART 2 §8, PART 3 §15 | Forgetting to sum log-likelihood over observations |
| "Estimate $p(f(x^*)>c)$ using samples" | `jnp.mean(f(samples) > c)` | Section 7.4 this notebook | Using $f$ samples when $y$ samples are needed (or vice versa) |
| "Explain how the MAP changes when $\alpha$ increases" | Prior pulls weights toward 0 more strongly → MAP closer to 0 | PART 1 §3.3 | $\alpha$ is precision, not variance - larger $\alpha$ = tighter prior |

</span>

##### **8.2 The five-step universal solve procedure**


For any exam question we are uncertain about, run through these steps in order:

**Step 1 - What is $y_n$?** Identify its domain. → Selects the likelihood (Section 1 this notebook).

**Step 2 - Is the posterior tractable?** Check: Gaussian likelihood + Gaussian prior + linear mean → yes. Otherwise no. → Selects the inference method (Section 2 this notebook).

**Step 3 - Are there latent variables to marginalise?** Discrete → sum. Gaussian → use identity. → (Section 3 this notebook).

**Step 4 - Find the formulas in PART 1–3.** Use the correct formula for the identified model and method.

**Step 5 - Answer the specific sub-question.** Use Section 7 for probabilities and intervals, Section 5 for kernel questions, Section 6 for VI.

##### **8.3 The most common sign of going wrong**


> If we are trying to use the GP posterior predictive formula but there is no $K$ matrix to invert yet → we are computing a **prior** predictive. Stop. Just evaluate $k(x^*,x^*)$.

> If we reach for the Bayesian linear regression closed-form formula but the likelihood is Bernoulli or Poisson → the posterior is **not Gaussian**. Stop. Use Laplace approximation or MCMC.

> If we are computing a credibility interval and the answer is 1.96 standard deviations but the question said 80% → we used the wrong $z$ value. 80% uses 1.282, not 1.96.

> If a marginalisation gives we a single Gaussian but the model had a mixture prior or a discrete latent variable → we collapsed something we should not have. The result should be a mixture.

> If our `log_target` returns a vector instead of a scalar → we forgot `.sum()` over the $N$ observations in the log-likelihood.

> If the exam says "prior probability" and our answer depends on data → we used the posterior. The prior is computed using only the prior parameters, no $y$.